<a href="https://colab.research.google.com/github/geologist8268/M-Nawaz/blob/master/WaPOR_data_Download_Single_ncfile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
!pip install --upgrade wapordl rioxarray xarray netCDF4 --quiet

import os
import glob
import xarray as xr
import rioxarray as rxr
from wapordl import wapor_map


In [8]:
# if you wanted to upload the geojson file;
#from google.colab import files
#uploaded = files.upload()

# 1) define region (remove comment (#) for the option you plan to use)
# define region using geojson file (if using a file stored in your google drive, add relative path here)
region = r"/content/drive/MyDrive/Colab_Notebooks/PRM.geojson"

In [9]:
# 2) define the variables and timeperiod to download
variables = ["L1-RET-M"] #, "L3-T-D","L3-NPP-D"]
period = ["2018-01-01", "2024-12-31"]
overview = "NONE"

In [10]:
# ------------------------------------------------------------------
# 2. Download rasters using WaPOR DL
# ------------------------------------------------------------------
for var in variables:
    folder = f"/content/output/{var}"
    os.makedirs(folder, exist_ok=True)

    if('-E' in var):
        unit = "day"
    elif('-D' in var):
        unit = "dekad"
    elif('-M' in var):
        unit = "month"
    elif ('-A' in var):
        unit = "year"
    else:
        unit = "none"

    fps = wapor_map(region, var, period, folder, separate_unscale=True, unit_conversion=unit)

In [11]:
# 3. Convert all TIFF files into a single NetCDF
# ------------------------------------------------------------------
tif_files = sorted(glob.glob(f"/content/output/{variables[0]}/*.tif"))

# Load and stack
rasters = []
dates = []
for f in tif_files:
    da = rxr.open_rasterio(f).squeeze(drop=True)
    rasters.append(da)

    # Extract date (assuming format L1-RET-M_YYYY_MM)
    date_str = os.path.basename(f).split("_")[-1].replace(".tif", "")
    dates.append(date_str)

ds = xr.concat(rasters, dim="time")
ds = ds.assign_coords(time=dates)


In [12]:
# 4. Export to NetCDF
# ------------------------------------------------------------------
nc_file = f"/content/WaPOR_{variables[0]}.nc"
ds.to_netcdf(nc_file)

print("NetCDF created:", nc_file)


NetCDF created: /content/WaPOR_L1-RET-M.nc


In [13]:
# 5. Download NetCDF
# ------------------------------------------------------------------
from google.colab import files
files.download(nc_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>